# RetailMind Churn Prediction - Inference

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.services.churn import predict

In [ ]:
DATA_PATH = PROJECT_ROOT / "data/Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
results = predict(df)

predictions = results["churn_prediction"].to_numpy()
probabilities = results["churn_probability"].to_numpy()

print("Predicted churn counts:")
print(results["churn_label"].value_counts())

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

y_true = df["Churn"].map({"Yes": 1, "No": 0}).to_numpy()
valid = ~pd.isna(y_true)

metrics = {
    "accuracy": accuracy_score(y_true[valid], predictions[valid]),
    "precision": precision_score(y_true[valid], predictions[valid], zero_division=0),
    "recall": recall_score(y_true[valid], predictions[valid], zero_division=0),
    "f1_score": f1_score(y_true[valid], predictions[valid], zero_division=0),
    "roc_auc": roc_auc_score(y_true[valid], probabilities[valid]),
}

pd.Series(metrics).round(4).to_frame("score")

In [ ]:
print(classification_report(y_true[valid], predictions[valid], target_names=["No Churn", "Churn"]))
print("Confusion matrix:")
print(confusion_matrix(y_true[valid], predictions[valid]))

In [ ]:
new_customer = df.iloc[[0]]
result = predict(new_customer).iloc[0]

print("Predicted churn:", result["churn_label"])
print("Churn probability:", round(result["churn_probability"], 4))